# Text Preprocessing

## Overview

Raw text is noisy, inconsistent, and high-dimensional. Preprocessing converts it into a clean, normalised form that downstream models can work with. The right preprocessing choices depend heavily on the task — aggressive cleaning that helps bag-of-words models can destroy semantic content needed by transformers.

**Standard preprocessing pipeline:**

```
Raw text
  → Lowercase
  → Remove HTML / special characters
  → Tokenise
  → Remove stopwords          ← task-dependent
  → Stem or lemmatise         ← task-dependent
  → Remove low/high-freq terms
  → Vectorise
```

**Key tools:** `re` (regex), `nltk`, `spacy`

**Ecological context:** processing scientific monitoring reports, site descriptions, and literature abstracts about water quality and restoration outcomes.

---

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Simulate ecological monitoring report text
texts = [
    "The restored riparian zone showed SIGNIFICANT improvement in water quality. Nitrate levels dropped by 45% after restoration!",
    "Site NORTH-04: elevated phosphorus (3.2 mg/L) detected. Possible run-off from upstream agricultural land-use.",
    "Species richness increased from 12 to 28 taxa following the wetland restoration project (2021-2023).",
    "No significant change observed at control sites. Water temperature remained stable at 14.2°C throughout monitoring.",
    "Buffer strips reduced nitrogen loading by ~30%. Results consistent with meta-analysis findings (Smith et al., 2019).",
    "Heavy rainfall event caused temporary turbidity spike. Data from 2022-03-15 flagged as potentially unreliable.",
    "Macroinvertebrate communities showed recovery: EPT richness (Ephemeroptera, Plecoptera, Trichoptera) increased 2-fold.",
    "pH levels normalised post-liming treatment. Previous readings <6.0 indicated acidification stress.",
]
print(f"{len(texts)} monitoring report snippets")
print(f"\nExample:\n{texts[0]}")

---
## Basic Cleaning with Regex

In [ ]:
def clean_text(text, lowercase=True, remove_punct=True,
               remove_numbers=False, remove_html=True):
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)
    if lowercase:
        text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Normalise whitespace
    text = re.sub(r'\s+', ' ', text)
    if remove_punct:
        text = text.translate(str.maketrans('', '', string.punctuation))
    if remove_numbers:
        text = re.sub(r'\b\d+\.?\d*\b', '', text)
    return text.strip()

cleaned = [clean_text(t) for t in texts]
print("Original:")
print(f"  {texts[0]}")
print("\nCleaned:")
print(f"  {cleaned[0]}")
print("\nAll cleaned texts:")
for t in cleaned:
    print(f"  {t[:80]}...")

---
## Tokenisation, Stopwords, Stemming, Lemmatisation

In [ ]:
try:
    import nltk
    # Download required data (silent if already present)
    for pkg in ['punkt','punkt_tab','stopwords','wordnet','averaged_perceptron_tagger']:
        try:
            nltk.download(pkg, quiet=True)
        except:
            pass
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer

    stop_words = set(stopwords.words('english'))
    stemmer    = PorterStemmer()
    lemmatizer = WordNetLemmatizer()

    def tokenise_and_normalise(text, use_lemma=True):
        tokens = word_tokenize(clean_text(text))
        tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
        if use_lemma:
            tokens = [lemmatizer.lemmatize(t) for t in tokens]
        else:
            tokens = [stemmer.stem(t) for t in tokens]
        return tokens

    example = texts[2]
    print(f"Original: {example}")
    print(f"\nLemmatised: {tokenise_and_normalise(example, use_lemma=True)}")
    print(f"Stemmed:    {tokenise_and_normalise(example, use_lemma=False)}")
    print("\nLemmatisation preserves real words; stemming is more aggressive (faster)")
except ImportError:
    print("pip install nltk")
    print("Key NLTK components:")
    print("  word_tokenize()      -> split text into tokens")
    print("  stopwords.words()    -> common words to remove")
    print("  WordNetLemmatizer()  -> 'running' -> 'run' (real word)")
    print("  PorterStemmer()      -> 'running' -> 'run' (may not be real word)")

---
## Vocabulary Analysis

In [ ]:
try:
    import nltk
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
    all_tokens = []
    for text in texts:
        tokens = word_tokenize(clean_text(text))
        all_tokens.extend([t for t in tokens if t.isalpha() and t not in stop_words])
    freq = Counter(all_tokens)
    vocab_size = len(freq)
    print(f"Vocabulary size: {vocab_size}")
    print(f"\nTop 20 terms:")
    for term, count in freq.most_common(20):
        print(f"  {term:20s}: {count}")
except ImportError:
    all_tokens = []
    for text in cleaned:
        all_tokens.extend(text.split())
    freq = Counter(all_tokens)
    print("Top 20 terms (no stopword removal):")
    for term, count in freq.most_common(20):
        print(f"  {term:20s}: {count}")
# Zipf's law visualisation
counts = sorted(freq.values(), reverse=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(range(1, len(counts)+1), counts, 'o-', ms=4, color='steelblue')
axes[0].set_xlabel('Rank'); axes[0].set_ylabel('Frequency')
axes[0].set_title("Word Frequency (Zipf's law)")
axes[1].loglog(range(1, len(counts)+1), counts, 'o-', ms=4, color='steelblue')
axes[1].set_xlabel('Rank (log)'); axes[1].set_ylabel('Frequency (log)')
axes[1].set_title("Log-Log: Linear = Zipf's law")
plt.tight_layout(); plt.show()

In [ ]:
# spaCy for richer tokenisation (named entities, POS tags)
try:
    import spacy
    try:
        nlp = spacy.load('en_core_web_sm')
    except OSError:
        print("spaCy model not found. Install with:")
        print("  python -m spacy download en_core_web_sm")
        raise
    doc = nlp(texts[0])
    print("spaCy tokenisation with POS and NER:")
    print(f"\nText: {texts[0]}")
    print("\nTokens with POS:")
    for token in doc:
        if not token.is_stop and not token.is_punct:
            print(f"  {token.text:20s} -> lemma: {token.lemma_:15s} POS: {token.pos_}")
    print("\nNamed entities:")
    for ent in doc.ents:
        print(f"  {ent.text:25s} -> {ent.label_}")
except ImportError:
    print("pip install spacy && python -m spacy download en_core_web_sm")
    print("spaCy provides: tokenisation, POS tagging, NER, dependency parsing")
    print("Preferred over NLTK for production NLP pipelines")

---

## Common Pitfalls

**1. Applying the same preprocessing to all tasks**  
Removing stopwords and punctuation is appropriate for bag-of-words models but destroys information for sentiment analysis ("not good" becomes "good") and sequence models. Always match preprocessing to the task: transformer models (BERT, etc.) typically need minimal preprocessing — just tokenisation.

**2. Stemming scientific or domain-specific text aggressively**  
Porter stemming truncates words to a common root, but "restoration" → "restor" and "restored" → "restor" may be appropriate while "Trichoptera" → "trichoptera" loses the taxonomic capitalisation signal. Lemmatisation is safer for domain text; consider domain-specific stopword lists.

**3. Fitting vocabulary statistics (IDF, stopwords frequency) on the full corpus including test documents**  
If IDF weights or vocabulary are computed over the whole dataset before train/test split, test document frequencies influence the vocabulary. Fit `TfidfVectorizer` (and any vocabulary-based step) on training documents only.

**4. Ignoring encoding issues when reading text files**  
Text files may be UTF-8, Latin-1, or Windows-1252. Reading a Latin-1 file as UTF-8 produces mojibake (garbled characters) that silently corrupts tokens. Always specify `encoding=` when reading text files and handle errors with `errors="replace"` or `errors="ignore"` after inspection.

**5. Treating token frequency as a direct proxy for importance**  
The most frequent tokens are often the least informative (stopwords, common verbs). Conversely, rare domain terms ("EPT richness", "Trichoptera") carry high diagnostic value. TF-IDF, rather than raw frequency, better captures token importance by penalising terms that are common across all documents.
---
*python_methods_library - Samantha McGarrigle*